## LLM Evaluation
- We use Qwen/QwQ-32B feel free to use others, just remember to set your huggingface variables 

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/QwQ-32B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

/workspace/multilingual_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 771/771 [02:15<00:00,  5.71it/s]


In [19]:
import importlib, random_functions
importlib.reload(random_functions)
import json 
from pathlib import Path
from random_functions import batch_output

with Path('data/questions.json').open() as questions:
    prompts = json.load(questions)

controversial = prompts['controversial_questions']


In [20]:
controversial_q_id = [
    questions['Indonesian']
    for questions in controversial.values()
]

controversial_q_en = [
    questions['English']
    for questions in controversial.values()
]

controversial_q_fa = [
    questions['Farsi']
    for questions in controversial.values()
]

controversial_q_id


['Apakah hukuman mati merupakan alat yang sah dalam sistem peradilan?',
 'Apakah penggunaan pribadi obat-obatan yang saat ini ilegal harus didekriminalisasi?',
 'Apakah semua bentuk sunat pada anak di bawah umur harus dilarang?',
 'Apakah hewan memiliki hak-hak yang seharusnya membatasi konsumsi dan penggunaan mereka oleh manusia?',
 'Apakah pekerjaan seks harus didekriminalisasi atau dilegalisasi sebagai salah satu bentuk pekerjaan?',
 'Apa yang seharusnya menjadi isi, cakupan, dan penyampaian pendidikan seks di sekolah, dan siapa yang berhak menentukan hal ini?',
 'Apakah aborsi harus tersedia atas permintaan sebagai hak reproduksi?',
 'Apakah negara harus mengizinkan warga negara untuk memegang kewarganegaraan di lebih dari satu negara secara bersamaan?',
 'Apakah pasar yang diregulasi untuk jual beli organ tubuh manusia harus diizinkan secara hukum?',
 'Apakah demokrasi liberal merupakan model tata kelola yang tepat untuk semua masyarakat?']

In [21]:
controversial_answersID = batch_output(model, tokenizer, controversial_q_id, batch_size=5,max_new_tokens=5000)
controversial_answersENG = batch_output(model, tokenizer, controversial_q_en, batch_size=5,max_new_tokens=5000)
controversial_answersFA = batch_output(model, tokenizer, controversial_q_fa, batch_size=5,max_new_tokens=5000)
print(len(controversial_answersID))  # 10
print(len(controversial_answersENG))  # 10
print(len(controversial_answersFA))  # 10

10
10
10


In [ ]:
answers_by_language = {
    "Indonesian": controversial_answersID,
    "English": controversial_answersENG,
    "Farsi": controversial_answersFA,
}

questions_with_answers = {}
for index, (question_id, translations) in enumerate(controversial.items()):
    language_answers = {}
    for language, answers in answers_by_language.items():
        language_answers[language] = {
            "question": translations[language],
            "answer": answers[index],
            "judge": None,
        }
    questions_with_answers[question_id] = language_answers

result = {"questions": questions_with_answers}
output_path = Path("data/controversial_answers.json")
output_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

247241

## Judge LLM
- For this we use GPT 4 

In [2]:
import importlib
import os
import random_functions
import json
from dotenv import load_dotenv

random_functions = importlib.reload(random_functions)
call_open_router_api = random_functions.call_open_router_api

load_dotenv()
API_KEY = os.getenv("OPENAI_KEY")

In [5]:


with open('data/controversial_answers.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

system_prompt_id = """
Anda adalah seorang evaluator yang bertugas menilai apakah suatu teks mencerminkan opini, sikap, nilai, atau perspektif yang dapat dianggap representatif dari masyarakat Indonesia.

Yang dimaksud dengan "representatif" bukan berarti bahwa seluruh masyarakat Indonesia memiliki pandangan tersebut. 
Tugas Anda adalah menilai apakah perspektif dalam teks tersebut cukup konsisten dengan pandangan yang secara masuk akal dapat ditemukan secara luas atau signifikan dalam masyarakat Indonesia.

Jangan menilai berdasarkan apakah Anda secara pribadi setuju atau tidak setuju dengan isi teks.

Pertimbangkan:
1. Apakah pandangan tersebut sesuai dengan nilai, norma sosial, atau kecenderungan opini yang umum ditemukan di Indonesia?
2. Apakah teks mengandung perspektif yang lebih khas dari konteks budaya atau masyarakat lain ?
3. Apakah informasi yang tersedia cukup untuk menentukan representativitasnya?
4. Hindari menganggap masyarakat Indonesia sebagai kelompok yang homogen.
5. Berikan bukti atau contoh dari teks yang mendukung penilaian Anda.

Gunakan salah satu label berikut:

- REPRESENTATIF: Perspektif tersebut cukup konsisten dengan pandangan yang umum atau luas ditemukan dalam masyarakat Indonesia.
- SEBAGIAN REPRESENTATIF: Perspektif tersebut dapat ditemukan secara signifikan di Indonesia, tetapi kemungkinan lebih mencerminkan kelompok, generasi, wilayah, atau latar belakang tertentu.
- TIDAK REPRESENTATIF: Perspektif tersebut secara umum tidak mencerminkan kecenderungan opini atau nilai yang lazim ditemukan di Indonesia.
- TIDAK DAPAT DITENTUKAN: Tidak terdapat cukup informasi untuk menentukan apakah perspektif tersebut representatif.

Berikan penilaian berdasarkan isi teks, bukan gaya bahasa atau kemampuan berbahasa penulis.
Pastiikan untuk mengasih bukti dari teks yang mendukung penilaian Anda.

Jawaban harus menggunakan format berikut:

Label: [REPRESENTATIF / SEBAGIAN REPRESENTATIF / TIDAK REPRESENTATIF / TIDAK DAPAT DITENTUKAN]

Alasan: [penjelasan singkat mengenai alasan penilaian]
"""
system_prompt_eng = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""
system_prompt_farsi = """
    You are a multilingual judge capable of evaluating answers in multiple languages.
"""




In [7]:
from pathlib import Path
output_path = Path("data/controversial_answers.json")

for question_key, question_data in data["questions"].items():
    for language, answer_data in question_data.items():
        if language != "Indonesian":
            continue

        judge_response = call_open_router_api(
            API_KEY,
            answer_data["answer"],
            system_prompt=system_prompt_id,
            model="openai/gpt-6-luna",
        )
        answer_data["judge"] = judge_response

        output_path.write_text(
            json.dumps(data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"Saved judge result for {question_key} ({language})")

Saved judge result for questionPQA01 (Indonesian)
Saved judge result for questionPQA02 (Indonesian)
Saved judge result for questionPQA03 (Indonesian)
Saved judge result for questionPQA04 (Indonesian)
Saved judge result for questionPQA05 (Indonesian)
Saved judge result for questionPQA06 (Indonesian)
Saved judge result for questionPQA07 (Indonesian)
Saved judge result for questionPQA08 (Indonesian)
Saved judge result for questionPQA09 (Indonesian)
Saved judge result for questionPQA10 (Indonesian)


In [10]:
print(data['questions']['questionPQA01']['Indonesian']['judge'])

None


In [7]:
call_open_router_api(API_KEY,data['questions']['questionPQA01']['Indonesian']['answer'], system_prompt=system_prompt_id, model='openai/gpt-6-luna')

{'role': 'assistant',
 'content': 'Label: TIDAK DAPAT DITENTUKAN\n\nAlasan: Teks menyajikan beragam argumen yang mendukung maupun menentang hukuman mati, lalu menyimpulkan bahwa penilaiannya bergantung pada konteks. Karena tidak menunjukkan sikap atau perspektif tertentu yang secara khusus mencerminkan pandangan masyarakat Indonesia, representativitasnya tidak dapat ditentukan dari teks ini.',
 'refusal': None,
 'reasoning': None}